# Notebook 07 - Extend the model to EchoNext (structural heart disease): Objective (b)
Thesis Sections 3.2.3, 3.4.4 and 3.5 applied to EchoNext.

The 1D-CNN, preprocessing and training settings are retrained on EchoNext with **one sigmoid output** (SHD yes/no) and one class weight in Equation 1. SHAP and Grad-CAM maps are aligned to the same 5 x 12 grid (Equation 2) and tested with the same faithfulness checks (Equation 3).

**Access first:** EchoNext is a restricted-access PhysioNet dataset. You need an approved PhysioNet account and a signed data use agreement (see the guide, Step 6.1). Its files must stay in your own storage and must never be committed to GitHub or shared.

EchoNext v1.1.1 provides NumPy arrays: `EchoNext_metadata_100k.csv` (label column `shd_moderate_or_greater_flag`, split column `split`) and `EchoNext_<split>_waveforms.npy` with shape (N, 1, 2500, 12): 10 s at 250 Hz, already median-filtered and normalised by the providers.

In [ ]:
# STEP 0 - Setup. Run this cell first, every time you open the notebook.
import os, sys, glob, json, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"   # the branch of your GitHub repository to use

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                       # click "Connect to Google Drive" and allow
    PROJECT = "/content/drive/MyDrive/FUTA-MTECH"        # everything you keep lives here
    REPO = "/content/FUTA-MTECH-PROJECT"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/Scientist47/FUTA-MTECH-PROJECT.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip -q install wfdb neurokit2 shap captum interpret-core
else:                                                    # running on your own computer
    PROJECT = os.environ.get("FUTA_PROJECT", os.path.abspath(".."))
    REPO = os.environ.get("FUTA_REPO", os.path.abspath(".."))

sys.path.insert(0, REPO)
os.chdir(REPO)
DATA = f"{PROJECT}/data"            # downloaded and processed data
RESULTS = f"{PROJECT}/results"      # tables, figures and trained models for Chapter 4
for d in (DATA, RESULTS, f"{RESULTS}/tables", f"{RESULTS}/figures", f"{RESULTS}/models"):
    os.makedirs(d, exist_ok=True)

# QUICK_TEST = True runs everything with tiny settings, to check the notebook works.
# Leave it False for your real thesis results.
QUICK_TEST = os.environ.get("FUTA_QUICK_TEST") == "1"

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
print("GPU available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only - see the guide, Step 1.3)")
print("Project folder:", PROJECT)

## Step 7.1 - Download EchoNext into your Drive (once; several GB)
Type your PhysioNet username and password when asked. They are not saved in the notebook.

In [ ]:
ECHO = os.environ.get("FUTA_ECHONEXT") or f"{DATA}/echonext"
if not glob.glob(f"{ECHO}/**/EchoNext_metadata_100k.csv", recursive=True):
    import getpass, subprocess
    os.makedirs(ECHO, exist_ok=True)
    user = input("PhysioNet username: "); pw = getpass.getpass("PhysioNet password: ")
    subprocess.run(["wget", "-r", "-N", "-c", "-np", "-nH", "--cut-dirs=3", "-q", "--show-progress",
                    "--user", user, "--password", pw, "-P", ECHO,
                    "https://physionet.org/files/echonext/1.1.1/"], check=True)
    del pw
ECHO = os.path.dirname(glob.glob(f"{ECHO}/**/EchoNext_metadata_100k.csv", recursive=True)[0])
print("EchoNext folder:", ECHO); print(sorted(os.listdir(ECHO)))

## Step 7.2 - Preprocess (Section 3.4.1): filter, resample 250 → 100 Hz, z-score with training statistics
EchoNext was recorded in the USA, where mains power is **60 Hz**, so the notch is set to 60 Hz here (the 50 Hz notch suits Nigerian and European recordings). State this in Chapter 4. The arrays are processed in chunks and saved as float16 to fit in Colab's memory.

In [ ]:
from preprocessing.echonext import load_split, LABEL, FS_ECHONEXT
from preprocessing.filters import highpass, notch, resample, fit_lead_stats, apply_zscore
NOTCH_HZ = 60
OUT = f"{DATA}/processed/echonext100"; os.makedirs(OUT, exist_ok=True)
def prep(x):
    x = notch(highpass(x, fs=FS_ECHONEXT), fs=FS_ECHONEXT, freq=NOTCH_HZ)
    return resample(x, FS_ECHONEXT, 100)
if not os.path.exists(f"{OUT}/X_test.npy"):
    filtered = {}
    for split in ["train", "val", "test"]:
        X, y, m = load_split(ECHO, split)
        Xf = np.concatenate([prep(X[i:i + 2000]) for i in range(0, len(X), 2000)])
        filtered[split] = (Xf, y, m)
        print(f"{split}: {Xf.shape}, SHD positives {int(y.sum()):,} ({100 * y.mean():.1f}%)")
    stats = fit_lead_stats(filtered["train"][0])        # training split only
    np.savez(f"{OUT}/lead_stats.npz", **stats)
    for split, (Xf, y, m) in filtered.items():
        np.save(f"{OUT}/X_{split}.npy", apply_zscore(Xf, stats).astype(np.float16))
        np.save(f"{OUT}/y_{split}.npy", y); m.to_csv(f"{OUT}/meta_{split}.csv", index=False)
    del filtered
data = {f"{a}_{s}": np.load(f"{OUT}/{a}_{s}.npy") for a in ["X", "y"] for s in ["train", "val"]}
for k, v in data.items(): print(k, v.shape, v.dtype)

### Table 4.13 - EchoNext dataset description

In [ ]:
rows = {}
for s in ["train", "val", "test"]:
    m = pd.read_csv(f"{OUT}/meta_{s}.csv")
    rows[s] = {"ECGs": len(m), "Patients": m.patient_key.nunique() if "patient_key" in m else "-",
               "SHD positive": int(m[LABEL].sum()), "SHD %": round(100 * m[LABEL].mean(), 1),
               "Mean age (standardised)": round(m.age_at_ecg.mean(), 2) if "age_at_ecg" in m else "-"}
table413 = pd.DataFrame(rows).T
table413.to_csv(f"{RESULTS}/tables/table4_13_echonext_splits.csv"); table413

## Step 7.3 - Train the 1D-CNN on EchoNext, five seeds
About 1-2 hours per seed on a T4. Kaggle (30 GPU hours a week) is a good alternative to Colab for this step.

In [ ]:
import yaml
from training.train import train, load_trained, predict
cfg = yaml.safe_load(open("training/configs/echonext_cnn1d.yaml"))
SEEDS = cfg["seeds"]
if QUICK_TEST:
    SEEDS = [0]; cfg.update(max_epochs=2, patience=1)
for seed in SEEDS:
    d = f"{RESULTS}/models/echonext_cnn1d/seed{seed}"
    if os.path.exists(f"{d}/summary.json"):
        print("seed", seed, "already trained"); continue
    train(cfg, seed, data, out_dir=f"{RESULTS}/models/echonext_cnn1d")

## Step 7.4 - Test-split performance (Table 4.14). Run once.

In [ ]:
from evaluation.classification_metrics import compute_metrics, metrics_with_ci, summarise_seeds
X_test, y_test = np.load(f"{OUT}/X_test.npy"), np.load(f"{OUT}/y_test.npy")
per_seed, ps = [], []
for seed in SEEDS:
    m, th = load_trained("cnn1d", 1, f"{RESULTS}/models/echonext_cnn1d/seed{seed}")
    p = predict(m, X_test); ps.append(p)
    per_seed.append(compute_metrics(y_test, p, th, ["SHD"]).loc["SHD"])
per_seed = pd.DataFrame(per_seed)
table414 = pd.DataFrame({k: ["{:.3f} ({:.3f}-{:.3f})".format(*summarise_seeds(per_seed[k]))] for k in per_seed.columns},
                        index=["1D-CNN, EchoNext SHD (mean of seeds, 95% CI)"])
p_mean = np.mean(ps, axis=0)
boot = metrics_with_ci(y_test, p_mean, th, ["SHD"], n_boot=200 if QUICK_TEST else 1000)
table414.to_csv(f"{RESULTS}/tables/table4_14_echonext_performance.csv"); display(table414); boot

## Step 7.5 - Explanations and faithfulness on EchoNext

In [ ]:
from xai.shap_explainer import explain_shap
from xai.gradcam import explain_gradcam
from xai.segment_alignment import batch_boundaries, align_batch, SEGMENTS
from evaluation import faithfulness as F
from scipy.stats import wilcoxon
from preprocessing.ptbxl import SUPERCLASSES, LEADS
model = load_trained("cnn1d", 1, f"{RESULTS}/models/echonext_cnn1d/seed0")[0]
rng = np.random.default_rng(0)
pos = np.where(y_test[:, 0] == 1)[0]
pick = np.sort(rng.choice(pos, min(20 if QUICK_TEST else 500, len(pos)), replace=False))
Xs = X_test[pick].astype(np.float32)
Ms = batch_boundaries(Xs, fs=100, lead=1)
bg = data["X_train"][rng.choice(len(data["X_train"]), 20 if QUICK_TEST else 100, replace=False)].astype(np.float32)
grids = {"shap": align_batch(explain_shap(model, Xs, bg, 0, nsamples=10 if QUICK_TEST else 50), Ms),
         "gradcam": align_batch(explain_gradcam(model, Xs, 0), Ms)}
np.savez_compressed(f"{RESULTS}/grids_echonext.npz", pick=pick, **grids)
rows = []
for method, g in grids.items():
    zeros = np.zeros(len(Xs), dtype=int)
    d = F.deletion_auc(model, Xs, Ms, g, zeros, step=6 if QUICK_TEST else 1)
    i = F.insertion_auc(model, Xs, Ms, g, zeros, step=6 if QUICK_TEST else 1)
    rows.append({"Method": method, "Deletion AUC": d["explanation"].mean(), "Deletion AUC random": d["random"].mean(),
                 "Deletion p": wilcoxon(d["explanation"], d["random"]).pvalue,
                 "Insertion AUC": i["explanation"].mean(), "Insertion AUC random": i["random"].mean(),
                 "Insertion p": wilcoxon(i["explanation"], i["random"]).pvalue})
table415 = pd.DataFrame(rows).round(4)
table415.to_csv(f"{RESULTS}/tables/table4_15_echonext_faithfulness.csv", index=False); table415

In [ ]:
# Randomisation check (cascading) for EchoNext
sub = slice(0, 10 if QUICK_TEST else 100)
for method in grids:
    fn = (lambda m, X: align_batch(explain_shap(m, X, bg, 0, nsamples=10 if QUICK_TEST else 50), Ms[sub])) if method == "shap" \
        else (lambda m, X: align_batch(explain_gradcam(m, X, 0), Ms[sub]))
    print(method); F.randomisation_check(model, fn, Xs[sub], Ms[sub])

## Step 7.6 - Figure 4.8 and Table 4.16: is the SHD profile like any PTB-XL superclass profile?
H2 expects EchoNext explanations to spread more evenly across segments. Compare the SHD profile with the PTB-XL profiles from notebook 04.

In [ ]:
from scipy.stats import spearmanr, entropy
ptb = dict(np.load(f"{RESULTS}/grids_test.npz"))
valid = lambda g: g[g.sum(axis=(1, 2)) > 0]
shd = valid(grids["shap"]).mean(0)
rows = [{"Profile": "EchoNext SHD", "Segment entropy": round(float(entropy(shd.sum(1))), 3),
         **{s: round(float(shd[k].sum()), 3) for k, s in enumerate(SEGMENTS)}, "Spearman with SHD": 1.0}]
for cls in SUPERCLASSES:
    p = valid(ptb[f"shap__cnn1d__{cls}"]).mean(0)
    rows.append({"Profile": f"PTB-XL {cls}", "Segment entropy": round(float(entropy(p.sum(1))), 3),
                 **{s: round(float(p[k].sum()), 3) for k, s in enumerate(SEGMENTS)},
                 "Spearman with SHD": round(float(spearmanr(shd.ravel(), p.ravel()).correlation), 3)})
table416 = pd.DataFrame(rows)
table416.to_csv(f"{RESULTS}/tables/table4_16_shd_vs_ptbxl_profiles.csv", index=False); display(table416)
plt.figure(figsize=(8, 3)); plt.imshow(shd, cmap="viridis", aspect="auto")
plt.xticks(range(12), LEADS); plt.yticks(range(5), SEGMENTS); plt.colorbar(label="share of attribution")
plt.title("EchoNext SHD: SHAP segment-by-lead profile (1D-CNN)")
plt.tight_layout(); plt.savefig(f"{RESULTS}/figures/fig4_8_shd_profile.png", dpi=300); plt.show()

Higher segment entropy = attribution spread more evenly across the five segments.

**Done.** For Chapter 4, Section 4.7 keep Tables 4.13-4.16 and Figure 4.8.